In [1]:
import os
import numpy as np
import cv2
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern, hog
from skimage.measure import shannon_entropy
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# Set the path to your main dataset folder
dataset_path = r"D:\Capston Project\Dataset_mansi_500"

In [3]:
# Common function to load images from a directory
def load_images(directory):
    images = []
    extensions = (".png", ".jpg", ".jpeg")
    for filename in os.listdir(directory):
        if filename.lower().endswith(extensions):
            img = cv2.imread(os.path.join(directory, filename), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(img)
    return images

# Define defect types
defect_types = [
    "Cutting_Marks", "Hot_tears_cracks", "Inclusion", "Porosity", "Scabs",
    "Shrink", "Surface_Roughness", "Veining", "Wrinkles_Folds_Coldshuts"
]

In [4]:
# Load images for each defect type
dataset = {defect: load_images(os.path.join(dataset_path, defect)) for defect in defect_types}

In [5]:
# Feature extraction functions
def extract_hog_features(image):
    hog_features, _ = hog(image, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=True)
    print(f"HOG feature dimension: {hog_features.shape}")
    return hog_features

def extract_lbp_features(image, n_points=24, radius=3):
    lbp = local_binary_pattern(image, n_points, radius, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    print(f"LBP feature dimension: {hist.shape}")
    return hist

def extract_glcm_features(image):
    glcm = graycomatrix(image, [1], [0, np.pi/4, np.pi/2, 3*np.pi/4], symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').mean()
    dissimilarity = graycoprops(glcm, 'dissimilarity').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    energy = graycoprops(glcm, 'energy').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    glcm_features = [contrast, dissimilarity, homogeneity, energy, correlation]
    print(f"GLCM feature dimension: {len(glcm_features)}")
    return glcm_features

def extract_haralick_features(image):
    glcm = graycomatrix(image, [1], [0], 256, symmetric=True, normed=True)
    haralick_features = [
        graycoprops(glcm, 'contrast')[0, 0],
        graycoprops(glcm, 'dissimilarity')[0, 0],
        graycoprops(glcm, 'homogeneity')[0, 0],
        graycoprops(glcm, 'energy')[0, 0],
        graycoprops(glcm, 'correlation')[0, 0],
        graycoprops(glcm, 'ASM')[0, 0]
    ]
    print(f"Haralick feature dimension: {len(haralick_features)}")
    return haralick_features

def extract_entropy(image):
    entropy = shannon_entropy(image)
    print(f"Entropy feature dimension: {1}")
    return entropy

# Common function to extract all features from an image
def extract_all_features(image):
    hog_feat = extract_hog_features(image)
    lbp_feat = extract_lbp_features(image)
    glcm_feat = extract_glcm_features(image)
    haralick_feat = extract_haralick_features(image)
    entropy = extract_entropy(image)
    all_features = np.concatenate([hog_feat, lbp_feat, glcm_feat, haralick_feat, [entropy]])
    print(f"Total feature dimension: {all_features.shape}")
    return all_features

# Extract features for a single image in each defect type to verify feature dimensions
for defect, images in dataset.items():
    if images:  # Ensure there is at least one image
        print(f"\nExtracting features for defect type: {defect}")
        extract_all_features(images[0])

# Extract features for all images in the dataset
features = {defect: [] for defect in defect_types}
for defect, images in dataset.items():
    for image in images:
        features[defect].append(extract_all_features(image))


Extracting features for defect type: Cutting_Marks
HOG feature dimension: (133956,)
LBP feature dimension: (26,)
GLCM feature dimension: 5
Haralick feature dimension: 6
Entropy feature dimension: 1
Total feature dimension: (133994,)

Extracting features for defect type: Hot_tears_cracks
HOG feature dimension: (133956,)
LBP feature dimension: (26,)
GLCM feature dimension: 5
Haralick feature dimension: 6
Entropy feature dimension: 1
Total feature dimension: (133994,)

Extracting features for defect type: Inclusion
HOG feature dimension: (133956,)
LBP feature dimension: (26,)
GLCM feature dimension: 5
Haralick feature dimension: 6
Entropy feature dimension: 1
Total feature dimension: (133994,)

Extracting features for defect type: Porosity
HOG feature dimension: (133956,)
LBP feature dimension: (26,)
GLCM feature dimension: 5
Haralick feature dimension: 6
Entropy feature dimension: 1
Total feature dimension: (133994,)

Extracting features for defect type: Scabs
HOG feature dimension: (13

In [6]:
# Prepare data for model training
X = []
y = []
label_mapping = {defect: idx for idx, defect in enumerate(defect_types)}

for defect, feat_array in features.items():
    X.extend(feat_array)
    y.extend([label_mapping[defect]] * len(feat_array))

X = np.array(X)
y = np.array(y)

In [7]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Logistic Regression
print("Logistic Regression:")
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
log_reg_acc = accuracy_score(y_test, y_pred_log_reg)
print(f"Accuracy: {log_reg_acc:.2f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_log_reg, target_names=defect_types))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log_reg))
print("\n" + "="*50 + "\n")

# Random Forest with varying n_estimators
print("Random Forest Classifier (n_estimators varying):")
rf_best_acc = 0
for n in range(100, 1001, 100):
    rf_model = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_model.fit(X_train, y_train)
    y_pred_rf = rf_model.predict(X_test)
    acc = accuracy_score(y_test, y_pred_rf)
    print(f"n_estimators={n}, Accuracy: {acc:.2f}")
    if acc > rf_best_acc:
        rf_best_acc = acc
        best_rf_model = rf_model
print(f"Best Random Forest Accuracy: {rf_best_acc:.2f}\n")

# SVM with RBF kernel and varying gamma
print("SVM Classifier with RBF Kernel (gamma varying):")
svm_rbf_best_acc = 0
for gamma in np.linspace(0, 1, 11):  # Gamma from 0 to 1 in steps
    svm_rbf = SVC(kernel='rbf', gamma=gamma)
    svm_rbf.fit(X_train, y_train)
    y_pred_svm_rbf = svm_rbf.predict(X_test)
    acc = accuracy_score(y_test, y_pred_svm_rbf)
    print(f"gamma={gamma:.2f}, Accuracy: {acc:.2f}")
    if acc > svm_rbf_best_acc:
        svm_rbf_best_acc = acc
        best_svm_rbf_model = svm_rbf
print(f"Best SVM RBF Accuracy: {svm_rbf_best_acc:.2f}\n")

# SVM with Polynomial kernel and varying degree
print("SVM Classifier with Polynomial Kernel (degree varying):")
svm_poly_best_acc = 0
for degree in range(1, 11):  # Degree from 1 to 10
    svm_poly = SVC(kernel='poly', degree=degree)
    svm_poly.fit(X_train, y_train)
    y_pred_svm_poly = svm_poly.predict(X_test)
    acc = accuracy_score(y_test, y_pred_svm_poly)
    print(f"degree={degree}, Accuracy: {acc:.2f}")
    if acc > svm_poly_best_acc:
        svm_poly_best_acc = acc
        best_svm_poly_model = svm_poly
print(f"Best SVM Polynomial Accuracy: {svm_poly_best_acc:.2f}\n")

Logistic Regression:


c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy: 0.45
Classification Report:
                          precision    recall  f1-score   support

           Cutting_Marks       0.49      0.48      0.49        89
        Hot_tears_cracks       0.52      0.52      0.52        54
               Inclusion       0.35      0.39      0.37        87
                Porosity       0.33      0.42      0.37        86
                   Scabs       0.36      0.32      0.34        47
                  Shrink       0.75      0.45      0.56        47
       Surface_Roughness       0.56      0.65      0.61       101
                 Veining       0.43      0.42      0.43        38
Wrinkles_Folds_Coldshuts       0.44      0.33      0.37        89

                accuracy                           0.45       638
               macro avg       0.47      0.44      0.45       638
            weighted avg       0.46      0.45      0.45       638

Confusion Matrix:
[[43  4 13 10  5  1  6  2  5]
 [ 3 28  3  4  1  1  2  2 10]
 [11  0 34 23  3  0  7 